# Course 5: NLP Applications
## Lecture 1: End-to-End NLP Workflows & Text Classification Application

---

### 🎯 Learning Objectives:
By the end of this lecture notebook, you will be able to:
1. Understand the full **End-to-End NLP Application Lifecycle** from raw text to production deployment.
2. Implement and compare **Classical Machine Learning (TF-IDF + Logistic Regression/LinearSVC)** and **Modern Deep Learning (Transformer / DistilBERT)** workflows.
3. Master text vectorization techniques (Bag-of-Words, n-grams, Subword Tokenization).
4. Evaluate NLP classification models using **Precision, Recall, F1-Score, Confusion Matrices**, and **Qualitative Error Analysis**.
5. Serialize and export trained pipelines (`joblib` and Hugging Face artifacts) ready for web serving.

---

### 📚 Course Architecture Overview

```
┌─────────────────┐     ┌──────────────────┐     ┌──────────────────────┐
│  Raw Text Data  │ ──> │ Text Preprocess  │ ──> │ Feature Extraction   │
│ (Reviews/Tweets)│     │ (Clean, Tokenize)│     │ (TF-IDF/Embeddings)  │
└─────────────────┘     └──────────────────┘     └──────────────────────┘
                                                             │
                                                             ▼
┌─────────────────┐     ┌──────────────────┐     ┌──────────────────────┐
│ Web UI / Deploy │ <── │  Model Export    │ <── │  Model Training &    │
│(Gradio/Streamlit│     │ (joblib/HF Hub)  │     │  Evaluation (Metrics)│
└─────────────────┘     └──────────────────┘     └──────────────────────┘
```


In [ ]:
# ==========================================
# Step 0: Environment Setup & Library Imports
# ==========================================
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

# Scikit-Learn tools for classical NLP
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# Transformers for Modern Deep Learning NLP
try:
    from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
    import torch
    print("✅ PyTorch & HuggingFace Transformers imported successfully!")
except ImportError:
    print("⚠️ Installing transformers and torch...")
    # In Jupyter/Colab: !pip install transformers torch


---
## Section 1: Creating a Realistic NLP Dataset
Let's build a realistic dataset representing **Customer Support Tickets & Product Reviews** categorized into:
- `Technical Support` (Bug reports, login issues, crashes)
- `Billing & Payment` (Refunds, invoices, card declines)
- `Product Feedback` (Feature requests, praise, UX complaints)
- `General Inquiry` (Hours of operation, contact info, shipping)


In [ ]:
# ==========================================
# Step 1: Create Sample Corpus for Application
# ==========================================
data = [
    # Technical Support
    ("The mobile application crashes every time I tap on the checkout button.", "Technical Support"),
    ("I cannot log in with my credentials. Getting error code 500.", "Technical Support"),
    ("The web page fails to load on Safari browser and freezes.", "Technical Support"),
    ("My sync failed and all my saved files are missing from the cloud dashboard.", "Technical Support"),
    ("Unable to reset password; the reset email link is broken or expired.", "Technical Support"),
    ("The audio output has strong static noise when joining calls on Android.", "Technical Support"),
    ("App keeps force closing after the latest version 2.4 update.", "Technical Support"),
    ("Database connection timed out while exporting report to PDF.", "Technical Support"),

    # Billing & Payment
    ("I was charged twice on my credit card for the annual subscription.", "Billing & Payment"),
    ("Where can I download the VAT tax invoice for my purchase last month?", "Billing & Payment"),
    ("I want to cancel my subscription and get a full refund immediately.", "Billing & Payment"),
    ("My payment failed via PayPal even though my bank account has sufficient balance.", "Billing & Payment"),
    ("How do I update my expired Mastercard on file?", "Billing & Payment"),
    ("You deducted $49 from my account without prior notification.", "Billing & Payment"),
    ("Requesting an invoice statement with my company tax registration ID.", "Billing & Payment"),
    ("Coupon code SUMMER2026 was not applied to my total order amount.", "Billing & Payment"),

    # Product Feedback
    ("I love the new dark mode theme! It is very easy on the eyes.", "Product Feedback"),
    ("Please add dark mode and customizable keyboard shortcuts in the next release.", "Product Feedback"),
    ("The new UI layout is confusing and takes too many clicks to reach settings.", "Product Feedback"),
    ("Fantastic user experience and super fast performance. Best tool ever!", "Product Feedback"),
    ("The search filter is lacking date range options; please improve search.", "Product Feedback"),
    ("Great tool overall, but the export options should include Markdown and CSV.", "Product Feedback"),
    ("The battery drain on iOS has increased noticeably with the new widgets.", "Product Feedback"),
    ("Really impressed with the speed improvements and sleek navigation menu.", "Product Feedback"),

    # General Inquiry
    ("What are your customer service working hours during public holidays?", "General Inquiry"),
    ("Do you offer international shipping to Egypt and the Middle East?", "General Inquiry"),
    ("Where is your regional office located in Cairo?", "General Inquiry"),
    ("Can I use this software on multiple devices simultaneously?", "General Inquiry"),
    ("Is there a student or academic discount available for university courses?", "General Inquiry"),
    ("What is your return policy for physical hardware accessories?", "General Inquiry"),
    ("Do you support single sign-on (SSO) for enterprise organization accounts?", "General Inquiry"),
    ("How long does standard postal delivery take to arrive?", "General Inquiry")
]

df = pd.DataFrame(data, columns=["text", "category"])
print(f"Total samples: {len(df)}")
print("\nClass Distribution:")
print(df['category'].value_counts())
df.head(6)


---
## Section 2: Text Preprocessing & Classical Vectorization (TF-IDF)

### Mathematical Foundation of TF-IDF:
Term Frequency-Inverse Document Frequency evaluates how important a word is to a document in a collection:

$$\text{TF}(t, d) = \frac{\text{count of } t \text{ in } d}{\text{total words in } d}$$

$$\text{IDF}(t, D) = \log \left( \frac{1 + |D|}{1 + |\{d \in D : t \in d\}|} \right) + 1$$

$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$

Let's split our dataset and build a scikit-learn pipeline.


In [ ]:
# ==========================================
# Step 2: Train-Test Split & Pipeline Setup
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], 
    df["category"], 
    test_size=0.25, 
    random_state=42, 
    stratify=df["category"]
)

print(f"Training samples: {len(X_train)} | Test samples: {len(X_test)}")

# Build an integrated Scikit-Learn Pipeline
# 1. TfidfVectorizer: Converts raw text to TF-IDF matrix with n-grams (1, 2)
# 2. LogisticRegression: Multi-class linear classifier
classic_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),        # Unigrams + Bigrams
        stop_words='english',      # Remove standard English stopwords
        sublinear_tf=True,         # Apply sublinear scaling (1 + log(tf))
        min_df=1                   # Minimum document frequency
    )),
    ('classifier', LogisticRegression(C=1.0, max_iter=200, random_state=42))
])

# Train pipeline
classic_pipeline.fit(X_train, y_train)
print("✅ Pipeline trained successfully!")


In [ ]:
# ==========================================
# Step 3: Model Evaluation (Metrics & Confusion Matrix)
# ==========================================
y_pred = classic_pipeline.predict(X_test)

print("📊 Classification Report:")
print(classification_report(y_test, y_pred))

# Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=classic_pipeline.classes_)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=classic_pipeline.classes_, 
            yticklabels=classic_pipeline.classes_)
plt.title("Confusion Matrix - Classical TF-IDF Classifier", fontsize=14, fontweight='bold')
plt.xlabel("Predicted Category", fontsize=12)
plt.ylabel("True Category", fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


---
## Section 3: Modern Pre-trained Transformers (Zero-Shot & DistilBERT)

While classical TF-IDF relies on exact keyword matching, **Transformers** leverage contextual embeddings via self-attention:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

This allows understanding **synonyms, semantic nuances, and out-of-vocabulary variations** without requiring thousands of labeled samples.


In [ ]:
# ==========================================
# Step 4: Hugging Face Zero-Shot & Sentiment Pipelines
# ==========================================
# Zero-shot classification can classify unseen text into arbitrary candidate labels!
candidate_labels = ["Technical Support", "Billing & Payment", "Product Feedback", "General Inquiry"]

classifier_hf = pipeline(
    "zero-shot-classification", 
    model="valhalla/distilbart-mnli-12-1" # Lightweight fast zero-shot model
)

sample_query = "The app keeps freezing on the payment screen and took my money without confirming."
result = classifier_hf(sample_query, candidate_labels)

print(f"Query: '{sample_query}'\n")
print(f"Top Predicted Category: {result['labels'][0]} ({result['scores'][0]:.2%})\n")
for label, score in zip(result['labels'], result['scores']):
    print(f" - {label:<20}: {score:.2%}")


In [ ]:
# ==========================================
# Sentiment Analysis with Hugging Face Pipeline
# ==========================================
sentiment_analyzer = pipeline(
    "sentiment-analysis", 
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

test_sentences = [
    "This software revolutionized our customer service workflow. Truly exceptional!",
    "Terrible customer service. Waiting for 3 days with zero reply.",
    "The product is decent, does the job as described."
]

print("🔍 Sentiment Analysis Results:")
for sentence in test_sentences:
    res = sentiment_analyzer(sentence)[0]
    print(f"Text: '{sentence}'")
    print(f"➡️  Sentiment: {res['label']} (Confidence: {res['score']:.2%})\n")


---
## Section 4: Exporting Models & Inference Wrapper for Web Applications

To deploy our NLP models into web applications (like **Gradio** or **Streamlit**), we must:
1. Serialize the trained pipeline using `joblib`.
2. Wrap the prediction logic in a clean, reusable Python function.


In [ ]:
# ==========================================
# Step 5: Exporting Pipeline & Production Wrapper
# ==========================================
os.makedirs("models", exist_ok=True)
model_path = "models/text_classifier.joblib"
joblib.dump(classic_pipeline, model_path)
print(f"✅ Pipeline exported to: {model_path}")

# Production Inference Function
def predict_support_ticket(text: str) -> dict:
    """
    Takes raw user ticket text and returns predicted category with probability scores.
    """
    if not text or len(text.strip()) == 0:
        return {"error": "Input text is empty"}
    
    # Load model
    loaded_model = joblib.load(model_path)
    
    # Predict probabilities
    probs = loaded_model.predict_proba([text])[0]
    classes = loaded_model.classes_
    
    # Format output dictionary
    prob_dict = {cls_name: float(round(prob, 4)) for cls_name, prob in zip(classes, probs)}
    sorted_probs = dict(sorted(prob_dict.items(), key=lambda item: item[1], reverse=True))
    
    top_category = list(sorted_probs.keys())[0]
    confidence = list(sorted_probs.values())[0]
    
    return {
        "top_category": top_category,
        "confidence": confidence,
        "all_probabilities": sorted_probs
    }

# Test the inference function
test_input = "Can I get a tax invoice with VAT for my company subscription?"
output = predict_support_ticket(test_input)
print("🧪 Test Inference Output:")
print(json.dumps(output, indent=2))


---
## Section 5: Key Takeaways & Comparison

| Dimension | Classical ML (TF-IDF + LogReg/SVM) | Pre-trained Transformers (DistilBERT/BERT) |
| :--- | :--- | :--- |
| **Training Speed** | ⚡ Extremely fast (milliseconds) | ⏳ Slower (requires GPU for large datasets) |
| **Inference Latency** | ⚡ Ultra-low (< 5ms on CPU) | 🕒 Moderate (20ms - 100ms on CPU) |
| **Memory Footprint** | 💾 Very small (< 10 MB) | 💾 Larger (250 MB - 1 GB+) |
| **Context Understanding**| ❌ Bag of Words (ignores word order) | ✅ Deep bidirectional contextual attention |
| **Sample Efficiency** | Needs labeled data for each task | ✅ Strong zero-shot & few-shot capabilities |

In the next lecture, we will explore **Text Summarization** and **Generative AI**! 🚀
